In [ ]:
import torch
import pandas as pd
from pathlib import Path

from src.prompt_manager import PromptManager
from src.data_manager import DataManager

from src import data_processing
import yaml
from src import paths
import hashlib

from src.analysis.reliability import ReliabilityAnalyzer

In [2]:
with open('../src/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

# active_analysis = full_config['active_analysis']
active_analysis = 'mcgill_qa_feedback'
model_vars = full_config['analyses'][active_analysis]['model_vars']
experimental_groups = model_vars['experimental_groups']

In [3]:
raw_df = pd.read_parquet(paths.RAW_DATA_DIR / f'{active_analysis}.parquet')

In [4]:
final_df = data_processing.get_analysis_ready_df(full_config=full_config,
                                                 active_analysis='mcgill_qa_feedback',
                                                 use_cache=False,
                                                 force_refresh=False)

Loading files for analysis mcgill_qa_feedback
🐢 Running full processing pipeline...
Finished loading experiment data
Found 0 experimental trials contaminated by garbage output.


C:\Users\Wouter Barter\Documents\AI_thesis\src\data_processing.py:278: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  weights_tensor = torch.tensor(weights, dtype=torch.float32)


In [6]:
analyzer = ReliabilityAnalyzer(final_df, group_cols=[
                               'model_name', 'prompt_id', 'dimension_name'], llm_rating_col='mean_rating')
analyzer.compute_reliability_gap(metric='spearman')

,model_name,prompt_id,dimension_name,metric_type,spearman_gap,spearman_hh,spearman_llm_avg
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,Spearman,0.000735,0.589539,0.588804
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,Spearman,0.006445,0.633026,0.626581
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,Spearman,0.024138,0.589539,0.565401
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,Spearman,0.024380,0.589520,0.565139
11,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Relevance,Spearman,0.068403,0.633026,0.564623
4,google/gemma-3-4b-it,261acd58fe,quality,Spearman,0.075549,0.561105,0.485556
5,google/gemma-3-4b-it,3fc3c2dce92c,Completeness,Spearman,0.084609,0.561105,0.476496
9,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Completeness,Spearman,0.090214,0.633026,0.542812
7,google/gemma-3-4b-it,3fc3c2dce92c,Relevance,Spearman,0.096643,0.561105,0.464461
10,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Directness,Spearman,0.098146,0.633026,0.534881


In [7]:
analyzer = ReliabilityAnalyzer(
    final_df, group_cols=['model_name', 'prompt_id', 'dimension_name'])
analyzer.analyze_calibration('human_disagreement')

,model_name,prompt_id,dimension_name,calibration_corr
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,0.306506
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,0.197388
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,0.175280
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,0.210163
4,google/gemma-3-4b-it,261acd58fe,quality,0.042260
5,google/gemma-3-4b-it,3fc3c2dce92c,Completeness,0.042098
6,google/gemma-3-4b-it,3fc3c2dce92c,Directness,0.017291
7,google/gemma-3-4b-it,3fc3c2dce92c,Relevance,-0.018862
8,meta-llama/Llama-3.2-3B-Instruct,261acd58fe,quality,-0.103952
9,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Completeness,0.106397


In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import scipy.stats as stats

In [9]:
full_model_df = final_df[['input_id', 'prompt_id', 'model_name', 'mean_human_rating',
                          'mode_rating', 'mean_rating', 'dimension_name', 'normalized_entropy']]

In [ ]:
pm = PromptManager(folder=Path(
    "../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK"))
pm.load_all()
prompt_hash_map = {key: item.description for key, item in pm.suites.items()}

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK
Scanning 2 suites from ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK...
Loaded 2 PromptSuites


{'3fc3c2dce92c': PromptSuite(id='3fc3c2dce92c', templates={'Relevance': PromptTemplate(id='f67646daad', name='naive_Relevance', dimension_name='Relevance', description='Naive prompt that measures the Relevance dimension', token_constraints=['1', '2', '3', '4'], tags=['naive', 'Relevance', 'baseline', 'scale_4'], system_message='You are an expert evaluator. Your task is to rate the overall Relevance of the Answer provided for the Question on a scale of 1 to 4, where 1 is the lowest Relevance and 4 is the highest Relevance.', user_message_template='Question:\n    {question}\n\n    Answer: \n    {answer}', _cached_constraint_ids=None), 'Completeness': PromptTemplate(id='606d61a7f7', name='naive_Completeness', dimension_name='Completeness', description='Naive prompt that measures the Completeness dimension', token_constraints=['1', '2', '3', '4'], tags=['naive', 'Completeness', 'baseline', 'scale_4'], system_message='You are an expert evaluator. Your task is to rate the overall Completenes

In [ ]:
models = {}

for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    if group_df['dimension_name'].unique()[0] == 'quality':
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df
    else:
        group_df_wide = group_df.pivot_table(index=['input_id', 'mean_human_rating', 'model_name'],
                                             columns='dimension_name',
                                             values=['normalized_entropy', 'mean_rating']).reset_index()
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_Relevance + mean_rating_Completeness + mean_rating_Directness"

    model = smf.ols(formula, data=group_df_wide).fit()

    models[str(f"{group_key[0]}_{prompt_hash_map[group_key[1]]}")] = model

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output


def interactive_dataframe_selector(data_dict, description="Select option:"):
    # label -> value pairs; value is the tuple key
    options = [k for k in data_dict.keys()]
    dropdown = widgets.Dropdown(
        options=options,
        description=description,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    def update_table(change):
        clear_output(wait=True)
        display(dropdown)
        display(data_dict[change.new].summary())

    dropdown.observe(update_table, names='value')
    display(dropdown)
    display(data_dict[dropdown.value].summary())

In [ ]:
interactive_dataframe_selector(models)

Dropdown(description='Select option:', index=2, layout=Layout(width='400px'), options=('Qwen/Qwen3-4B-Instruct…

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      mean_human_rating   R-squared:                       0.262
Model:                            OLS   Adj. R-squared:                  0.262
Method:                 Least Squares   F-statistic:                     2013.
Date:                Mon, 02 Feb 2026   Prob (F-statistic):               0.00
Time:                        08:42:05   Log-Likelihood:                -7688.5
No. Observations:                5662   AIC:                         1.538e+04
Df Residuals:                    5660   BIC:                         1.539e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.2281      0.052      4.397      0.000       0.126       0.330
mean_rating     0.6845      0.015     44.866      0.000       0.655       0.714
==============================================================================
Omnibus:                      362.861   Durbin-Watson:                   1.912
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              150.551
Skew:                          -0.166   Prob(JB):                     2.03e-33
Kurtosis:                       2.273   Cond. No.                         15.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""